# Análisis Semántico con BERT (bert-base-uncased)

Las canciones del corpus están en **inglés**, por lo que se usa `bert-base-uncased` en lugar de BETO.
Se cubren todos los análisis requeridos por el Criterio 3 del proyecto:

1. Carga del modelo desde HuggingFace
2. Polisemia contextual (misma palabra → embedding distinto por contexto/género)
3. Búsqueda semántica de canciones (vector [CLS])
4. Masked Language Model por género
5. Guardado de embeddings en MongoDB


## 1. Imports y configuración

In [1]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.spatial.distance import cosine
import torch
from transformers import BertTokenizer, BertModel, pipeline
import sys
sys.path.append(os.path.abspath('../../../..'))
from src.data.mongo_storage import _get_default_collection, guardar_embeddings

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FIGS   = Path('../../../..') / 'data' / 'figures'
MODELS = Path('../../../..') / 'data' / 'models'
for d in [FIGS, MODELS]: d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.facecolor':'white','axes.grid':True,'grid.alpha':0.3,'font.size':11})
PALETTE = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2',
           '#937860','#DA8BC3','#8C8C8C','#CCB974','#64B5CD']
print(f'✓ Imports OK — dispositivo: {DEVICE}')

✓ Imports OK — dispositivo: cpu


## 2. Cargar corpus desde MongoDB

In [2]:
col = _get_default_collection()

docs = list(col.find(
    {'Lyrics': {'$ne': None}},
    {'_id': 1, 'Song': 1, 'Artist': 1, 'Genre': 1,
     'Song year': 1, 'Lyrics': 1, 'Language': 1}
))
df = pd.DataFrame(docs)
df = df.dropna(subset=['Lyrics', 'Genre']).copy()
df['Lyrics'] = df['Lyrics'].astype(str)
df = df[df['Lyrics'].str.len() > 50].reset_index(drop=True)

# Solo géneros con >= 20 canciones
conteo = df['Genre'].value_counts()
generos_validos = conteo[conteo >= 20].index.tolist()
df = df[df['Genre'].isin(generos_validos)].reset_index(drop=True)

print(f'✓ {len(df):,} canciones cargadas desde MongoDB | {df["Genre"].nunique()} géneros')
print(df['Genre'].value_counts().to_string())

✓ 7,935 canciones cargadas desde MongoDB | 10 géneros
Genre
Rock          1410
Pop           1110
Hip-Hop        960
Country        810
Metal          810
Jazz           660
Electronic     660
Indie          510
R&B            510
Folk           495


## 3. Preparar corpus completo para BERT

Se procesan **todas** las canciones del corpus, sin muestreo.
El sistema de caché en disco evita recalcular si ya existen los embeddings.

In [3]:
# Sin muestreo: se usa el corpus completo
df_sample = df.copy()   # df_sample mantiene el nombre para compatibilidad con el resto del notebook
LYRICS = df_sample['Lyrics'].tolist()
GENRES = df_sample['Genre'].tolist()
print(f'✓ Corpus completo para BERT: {len(df_sample):,} canciones')
print(df_sample['Genre'].value_counts().to_string())

✓ Corpus completo para BERT: 7,935 canciones
Genre
Rock          1410
Pop           1110
Hip-Hop        960
Country        810
Metal          810
Jazz           660
Electronic     660
Indie          510
R&B            510
Folk           495


## 4. Cargar modelo BERT (inglés)

Las canciones del corpus están en **inglés**, por lo que se usa `bert-base-uncased`.
Si en algún momento el corpus incluyera canciones en español, se cambiaría a `dccuchile/bert-base-spanish-wwm-cased` (BETO).

In [4]:
# bert-base-uncased → BERT entrenado en inglés (Wikipedia + BookCorpus)
# Alternativa española (BETO): 'dccuchile/bert-base-spanish-wwm-cased'
MODEL_NAME = 'bert-base-uncased'

print(f'Cargando {MODEL_NAME}...')
print('(Primera vez: descarga ~440 MB)\n')

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model_bert = BertModel.from_pretrained(MODEL_NAME).to(DEVICE)
model_bert.eval()

print(f'✓ BERT cargado en {DEVICE}')
print(f'   Parámetros  : {sum(p.numel() for p in model_bert.parameters()):,}')
print(f'   Dim. output : {model_bert.config.hidden_size}')

Cargando bert-base-uncased...
(Primera vez: descarga ~440 MB)



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7082.63it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ BERT cargado en cpu
   Parámetros  : 109,482,240
   Dim. output : 768


## 5. Funciones de embedding

In [5]:
def bert_embedding(text: str) -> np.ndarray:
    """Embedding BERT — vector [CLS] del último hidden state."""
    inputs = tokenizer(
        text[:1000], return_tensors='pt',
        truncation=True, max_length=512, padding='max_length',
    ).to(DEVICE)
    with torch.no_grad():
        out = model_bert(**inputs)
    return out.last_hidden_state[:, 0, :].squeeze().cpu().numpy()


def bert_embeddings_batch(texts: list, batch_size: int = 16) -> np.ndarray:
    """Genera embeddings en batches con barra de progreso."""
    result = []
    n = len(texts)
    for i in range(0, n, batch_size):
        batch = texts[i:i + batch_size]
        result.extend([bert_embedding(t) for t in batch])
        done = min(i + batch_size, n)
        bar  = '█' * int(done/n*25) + '░' * (25 - int(done/n*25))
        print(f'  [{bar}] {done}/{n}', end='\r')
    print()
    return np.array(result)


def word_contextual_embedding(sentence: str, word: str) -> np.ndarray:
    """Embedding contextual de una palabra dentro de su oración."""
    inputs = tokenizer(
        sentence, return_tensors='pt', truncation=True, max_length=128
    ).to(DEVICE)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    # Busca todos los sub-tokens que forman la palabra
    idxs = [i for i, tok in enumerate(tokens)
            if word.lower() in tok.lower().replace('##', '')]
    with torch.no_grad():
        out = model_bert(**inputs)
    hidden = out.last_hidden_state[0].cpu().numpy()
    return hidden[idxs].mean(axis=0) if idxs else hidden.mean(axis=0)


# Prueba rápida
e = bert_embedding('this is a test song about love and night')
print(f'✓ Funciones listas — embedding shape: {e.shape}')

✓ Funciones listas — embedding shape: (768,)


## 6. Generar embeddings BERT y guardar en MongoDB

In [6]:
BERT_FILE = MODELS / 'bert_embeddings.npy'

# Caché en disco: si ya existe y el tamaño coincide, se reutiliza
if BERT_FILE.exists():
    bert_embs = np.load(BERT_FILE)
    if len(bert_embs) == len(df_sample):
        print(f'✓ Embeddings cargados desde caché: {bert_embs.shape}')
    else:
        BERT_FILE.unlink()
        print(f'Caché desactualizada ({len(bert_embs)} vs {len(df_sample)}), recalculando...')

if not BERT_FILE.exists():
    print(f'Generando embeddings BERT para {len(LYRICS):,} canciones...')
    print(f'(Estimado en CPU ~{len(LYRICS)//60} min | GPU ~{len(LYRICS)//600} min)')
    bert_embs = bert_embeddings_batch(LYRICS, batch_size=16)
    np.save(BERT_FILE, bert_embs)
    print(f'✓ Embeddings guardados en disco: {bert_embs.shape}')

# Guardar en MongoDB (beto_cls) sin pisar word2vec_avg
actualizados = 0
for i, (_, row) in enumerate(df_sample.iterrows()):
    doc = col.find_one({'_id': row['_id']}, {'embeddings': 1})
    w2v_avg = []
    if doc and doc.get('embeddings') and doc['embeddings'].get('word2vec_avg'):
        w2v_avg = doc['embeddings']['word2vec_avg']
    ok = guardar_embeddings(
        song_id=row['_id'],
        word2vec_avg=w2v_avg,
        beto_cls=bert_embs[i].tolist(),
    )
    if ok: actualizados += 1
    if (i + 1) % 500 == 0:
        print(f'  MongoDB: {i+1}/{len(df_sample)} guardadas...', end='\r')

print(f'\n✓ bert_cls guardado en MongoDB: {actualizados}/{len(df_sample)} canciones')
print(f'Shape final: {bert_embs.shape}')

Generando embeddings BERT para 7,935 canciones...
(Estimado en CPU ~132 min | GPU ~13 min)
  [█████████████████████████] 7935/7935
✓ Embeddings guardados en disco: (7935, 768)
  MongoDB: 7500/7935 guardadas...
✓ bert_cls guardado en MongoDB: 7935/7935 canciones
Shape final: (7935, 768)


## 7. Análisis de polisemia contextual

Se seleccionan 5 palabras polisémicas comunes en letras musicales en inglés.
BERT genera un vector **distinto** para la misma palabra según el contexto y género,
demostrando que la representación es **contextual** (a diferencia de Word2Vec, que asigna un único vector por palabra).

In [7]:
# 5 palabras polisémicas con 5 contextos por género
POLISEMIA = {
    'fire': [
        ('Rock',       'the guitar is on fire tonight we burn the stage with our sound'),
        ('Pop',        'your love set fire to my heart every time you smile at me'),
        ('Hip-Hop',    'we fire back without retreating the streets keep us ready for war'),
        ('Metal',      'fire and brimstone rain from hell consuming every last piece of light'),
        ('Electronic', 'fire up the synth drop the bass hard and let the crowd explode'),
    ],
    'heart': [
        ('Pop',        'my heart beats faster when you hold me close under the stars'),
        ('Rock',       'heart of gold soul of iron we fight until the bitter end'),
        ('Hip-Hop',    'put your heart into the grind every single day no days off'),
        ('Country',    'roads that lead me back to where my heart has always belonged'),
        ('R&B',        'your heart is all I need tonight so please just stay with me'),
    ],
    'night': [
        ('Jazz',       'feeling blue tonight the trumpet cries for all my lost loves gone'),
        ('Electronic', 'dancing all night long under neon lights to the rhythm of bass'),
        ('Folk',       'quiet night beside the river willows whisper your name so softly'),
        ('Metal',      'darkness falls into the night consuming everything that lives below now'),
        ('Pop',        'dancing with you through the night until the sun comes up tomorrow'),
    ],
    'road': [
        ('Country',    'long road winding home through the fields where my childhood memories live'),
        ('Rock',       'hit the road and never look back freedom calling from far away'),
        ('Folk',       'walking the road alone searching for a place I can call home'),
        ('Hip-Hop',    'came up from a hard road now the whole crew eats well tonight'),
        ('Indie',      'the road ahead is blurry and uncertain yet I keep on moving forward'),
    ],
    'light': [
        ('Metal',      'the light fades into eternal darkness shadows swallow every soul alive now'),
        ('Pop',        'you are the light that brightens up my day whenever you are near'),
        ('Electronic', 'strobing light floods the dancefloor as the beat drops hard below us'),
        ('Folk',       'morning light streams through the window birds begin to sing outside softly'),
        ('R&B',        'you shine a light on the darkest corners of my broken wounded heart'),
    ],
}

# Filtrar géneros que realmente están en el corpus
POLISEMIA = {
    word: [(g, s) for g, s in contexts if g in generos_validos]
    for word, contexts in POLISEMIA.items()
}
POLISEMIA = {w: c for w, c in POLISEMIA.items() if len(c) >= 3}

print('Calculando embeddings contextuales...')
polisemia_embs = {}
for palabra, contextos in POLISEMIA.items():
    polisemia_embs[palabra] = []
    for genero, oracion in contextos:
        emb = word_contextual_embedding(oracion, palabra)
        polisemia_embs[palabra].append({'genero': genero, 'oracion': oracion, 'emb': emb})
    print(f'  ✓ "{palabra}" — {len(contextos)} contextos')

Calculando embeddings contextuales...
  ✓ "fire" — 5 contextos
  ✓ "heart" — 5 contextos
  ✓ "night" — 5 contextos
  ✓ "road" — 5 contextos
  ✓ "light" — 5 contextos


In [10]:
# 1. Borra la carpeta del entorno actual (fuerza el borrado)
Remove-Item -Recurse -Force .venv

# 2. Crea el entorno de nuevo desde cero
python -m venv .venv

# 3. Activa el entorno
.\.venv\Scripts\Activate.ps1

SyntaxError: invalid syntax (117647385.py, line 5)

In [8]:
n_words = len(polisemia_embs)
fig, axes = plt.subplots(1, n_words, figsize=(n_words * 3.8, 5))
if n_words == 1: axes = [axes]

for ax, (palabra, resultados) in zip(axes, polisemia_embs.items()):
    n = len(resultados)
    sim = np.zeros((n, n))
    etiquetas = [r['genero'] for r in resultados]
    for i in range(n):
        for j in range(n):
            sim[i, j] = 1 - cosine(resultados[i]['emb'], resultados[j]['emb'])
    im = ax.imshow(sim, cmap='RdYlGn', vmin=0.4, vmax=1.0)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(etiquetas, rotation=35, ha='right', fontsize=8)
    ax.set_yticklabels(etiquetas, fontsize=8)
    ax.set_title(f'"{palabra}"', fontweight='bold', fontsize=11)
    for i in range(n):
        for j in range(n):
            color = 'white' if sim[i,j] < 0.7 else 'black'
            ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center',
                    fontsize=7.5, color=color, fontweight='bold')

plt.colorbar(im, ax=axes[-1], label='Similitud coseno', shrink=0.8)
plt.suptitle(
    'Polisemia contextual con BERT (bert-base-uncased)\n'
    'La misma palabra tiene representaciones distintas según el contexto y género',
    fontsize=12, fontweight='bold', y=1.05
)
plt.tight_layout()
plt.savefig(FIGS / 'polisemia_bert.png', dpi=130, bbox_inches='tight')
plt.show()

# Análisis cuantitativo: varianza inter-contexto por palabra
print('\n=== Varianza inter-contexto (cuanto más alto → más polisemia capturada) ===')
for palabra, resultados in polisemia_embs.items():
    embs = np.array([r['emb'] for r in resultados])
    pares_sim = []
    n = len(embs)
    for i in range(n):
        for j in range(i+1, n):
            pares_sim.append(1 - cosine(embs[i], embs[j]))
    print(f'  "{palabra}": sim_media={np.mean(pares_sim):.4f} | '
          f'sim_min={np.min(pares_sim):.4f} | sim_max={np.max(pares_sim):.4f}')

ModuleNotFoundError: No module named 'matplotlib_inline'

## 8. Búsqueda semántica de canciones

Dado un texto de consulta, se encuentra las canciones más similares semánticamente
usando la distancia coseno entre embeddings BERT [CLS].

In [ ]:
def busqueda_semantica(consulta: str, top_k: int = 5) -> pd.DataFrame:
    """Encuentra las top_k canciones más similares a la consulta."""
    q_emb = bert_embedding(consulta)
    sims  = np.array([1 - cosine(q_emb, e) for e in bert_embs])
    top_idx = np.argsort(sims)[::-1][:top_k]
    res = df_sample.iloc[top_idx].copy()
    res['Similitud'] = sims[top_idx].round(4)
    return res[['Song', 'Artist', 'Genre', 'Similitud']]


CONSULTAS = [
    ('rebellious energy distorted guitar anthem loud crowd',      '→ Expected: Rock/Metal'),
    ('romantic love dancing together through the night',          '→ Expected: Pop/R&B'),
    ('street money power respect hustle grind success',           '→ Expected: Hip-Hop'),
    ('darkness destruction chaos war evil consuming all',         '→ Expected: Metal'),
    ('introspective melancholy acoustic quiet solitude',          '→ Expected: Indie/Folk'),
    ('jazzy blues trumpet saxophone soul late night',             '→ Expected: Jazz'),
    ('electronic bass drop synth club dance floor',               '→ Expected: Electronic'),
]
# Filtrar consultas a géneros presentes en el corpus

print('=== Búsqueda Semántica con BERT ===\n')
for consulta, esperado in CONSULTAS:
    print(f'🔍 "{consulta}"')
    print(f'   {esperado}')
    resultados = busqueda_semantica(consulta, top_k=3)
    for _, r in resultados.iterrows():
        print(f'   [{r["Genre"]:12s}] {r["Similitud"]:.4f}  '
              f'{str(r["Song"])[:35]:35s} – {r["Artist"]}')
    print()

### Evaluación de precisión de la búsqueda semántica

In [ ]:
# Precision@K: para cada canción de la muestra, ¿las K más similares son del mismo género?
K = 5
precision_por_genero = {g: [] for g in generos_validos}

for idx in range(len(bert_embs)):
    sims = np.array([1 - cosine(bert_embs[idx], bert_embs[j])
                     for j in range(len(bert_embs))])
    sims[idx] = -1  # excluir la propia canción
    top_k_idx = np.argsort(sims)[::-1][:K]
    genero_query = df_sample.iloc[idx]['Genre']
    hits = sum(df_sample.iloc[j]['Genre'] == genero_query for j in top_k_idx)
    precision_por_genero[genero_query].append(hits / K)

print(f'=== Precision@{K} por género (búsqueda semántica BERT) ===')
precisions = []
for genero, vals in sorted(precision_por_genero.items()):
    if vals:
        p = np.mean(vals)
        precisions.append(p)
        print(f'  {genero:<12} : {p:.4f}')
print(f'  {"PROMEDIO":<12} : {np.mean(precisions):.4f}')

## 9. Masked Language Model (Fill-Mask) por género

Se usa la capacidad de predicción de palabras enmascaradas de BERT para analizar
cómo el modelo completa frases típicas de cada género. Las predicciones reflejan
el vocabulario propio del dominio musical en inglés.

In [ ]:
print('Cargando pipeline fill-mask...')
mlm = pipeline('fill-mask', model=MODEL_NAME,
               device=0 if torch.cuda.is_available() else -1)
print('✓ Pipeline listo\n')

# Una frase representativa por género con [MASK] en posición clave
MLM_FRASES = {
    'Rock':       'We will break the [MASK] and rise above the pain tonight loud',
    'Pop':        'Your love makes my [MASK] beat faster every time we are together',
    'Hip-Hop':    'We started from the [MASK] and now the whole crew is on top',
    'Metal':      'The [MASK] descends upon the earth consuming every last bit of light',
    'Electronic': 'Drop the [MASK] and let the whole crowd move to the beat now',
    'Jazz':       'Late night [MASK] fills the smoky room with memories of better days',
    'Country':    'Back home on the [MASK] where the sky is wide and hearts are free',
    'Folk':       'Sitting by the [MASK] singing songs our grandparents used to know well',
    'Indie':      'Lost in my own [MASK] searching for a meaning in this world today',
    'R&B':        'Tonight I want to [MASK] you close and never let you go away',
}
# Filtrar a los géneros presentes en el corpus
MLM_FRASES = {g: f for g, f in MLM_FRASES.items() if g in generos_validos}

mlm_resultados = {}
print('=== Predicciones BERT MLM por Género ===')
for genero, frase in MLM_FRASES.items():
    preds = mlm(frase, top_k=5)
    palabras = [p['token_str'].strip() for p in preds]
    probs    = [p['score'] for p in preds]
    mlm_resultados[genero] = {'frase': frase, 'palabras': palabras, 'probs': probs}
    print(f'  {genero:<12} → {palabras}')

In [ ]:
n_generos = len(mlm_resultados)
cols = 2; rows = (n_generos + 1) // 2
fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3.5))
axes = axes.flatten()
colores_genero = dict(zip(mlm_resultados.keys(), PALETTE))

for ax, (genero, datos) in zip(axes, mlm_resultados.items()):
    palabras = datos['palabras'][:5]
    probs    = datos['probs'][:5]
    color    = colores_genero.get(genero, '#999')
    bars = ax.barh(palabras[::-1], probs[::-1], color=color, edgecolor='white', height=0.6)
    ax.set_xlim(0, max(probs) * 1.4)
    frase_corta = datos['frase'].replace('[MASK]', '___')[:65] + '...'
    ax.set_title(f'{genero}\n"{frase_corta}"', fontsize=9, fontweight='bold')
    ax.set_xlabel('Probability', fontsize=9)
    for bar, prob in zip(bars, probs[::-1]):
        ax.text(bar.get_width() + max(probs)*0.02,
                bar.get_y() + bar.get_height()/2,
                f'{prob:.3f}', va='center', fontsize=9)

for ax in axes[len(mlm_resultados):]:
    ax.set_visible(False)

plt.suptitle('BERT Masked LM — Top 5 predicciones por Género (bert-base-uncased)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'mlm_bert_generos.png', dpi=130, bbox_inches='tight')
plt.show()

## 10. Similitud entre géneros con BERT

Centroides de los embeddings BERT por género y matriz de similitud coseno.
A diferencia de Word2Vec, los embeddings BERT son contextuales y capturan
mejor las diferencias semánticas entre géneros.

In [ ]:
# Centroide BERT por género
generos_sample = df_sample['Genre'].unique().tolist()
centroides_bert = {}
for genero in generos_sample:
    idxs = df_sample[df_sample['Genre'] == genero].index.tolist()
    # Mapear a posición en bert_embs (puede diferir si se reindexó)
    pos  = [df_sample.index.get_loc(i) for i in idxs if i in df_sample.index]
    if pos:
        centroides_bert[genero] = bert_embs[pos].mean(axis=0)

generos_bert = list(centroides_bert.keys())
n = len(generos_bert)
sim_bert = np.zeros((n, n))
for i, g1 in enumerate(generos_bert):
    for j, g2 in enumerate(generos_bert):
        sim_bert[i, j] = 1 - cosine(centroides_bert[g1], centroides_bert[g2])

vmin_bert = sim_bert[~np.eye(n, dtype=bool)].min()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(sim_bert, cmap='YlOrRd', vmin=vmin_bert, vmax=1.0)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(generos_bert, rotation=40, ha='right')
ax.set_yticklabels(generos_bert)
ax.set_title('Similitud Coseno entre Géneros (BERT bert-base-uncased)',
             fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Similitud coseno')
for i in range(n):
    for j in range(n):
        color = 'white' if sim_bert[i,j] > 0.75 else 'black'
        ax.text(j, i, f'{sim_bert[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color=color, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGS / 'similitud_generos_bert.png', dpi=130, bbox_inches='tight')
plt.show()

pares = [(generos_bert[i], generos_bert[j], sim_bert[i,j])
         for i in range(n) for j in range(i+1, n)]
pares.sort(key=lambda x: -x[2])
print('Top 3 pares más SIMILARES (BERT):')
for g1, g2, s in pares[:3]: print(f'  {g1} ↔ {g2}: {s:.4f}')
print('\nTop 3 pares más DIFERENTES (BERT):')
for g1, g2, s in pares[-3:]: print(f'  {g1} ↔ {g2}: {s:.4f}')

## 11. Word2Vec vs BERT — diferencias semánticas

Se compara cualitativamente cómo cada modelo representa la misma palabra.
Word2Vec: un único vector estático. BERT: vector diferente según el contexto.

In [ ]:
print('=== Comparación Word2Vec (estático) vs BERT (contextual) ===\n')

COMPARACION_FRASES = {
    'fire': [
        'The guitar solo was on fire and the crowd went completely wild tonight',
        'Open fire on the enemy no mercy in this brutal endless war zone',
        'Sitting by the fire watching embers fade into the quiet cold night',
    ],
    'light': [
        'You are the light of my life shining bright in all the darkness',
        'The strobe light flashed as the DJ dropped the hardest bass ever',
        'Travel light and leave behind all the baggage weighing down your soul',
    ],
    'heart': [
        'She broke my heart and left me standing in the cold dark rain',
        'He had the heart of a lion fearless in every single battle fought',
        'Put your heart and soul into the music let it all out now',
    ],
}

for palabra, frases in COMPARACION_FRASES.items():
    print(f'--- "{palabra}" ---')
    embs_bert_word = [word_contextual_embedding(f, palabra) for f in frases]
    print('  BERT — similitud entre contextos distintos:')
    for i in range(len(frases)):
        for j in range(i+1, len(frases)):
            s = 1 - cosine(embs_bert_word[i], embs_bert_word[j])
            ctx_i = frases[i][:40] + '...'
            ctx_j = frases[j][:40] + '...'
            print(f'    [{ctx_i}] ↔ [{ctx_j}] = {s:.4f}')
    print('  (Word2Vec siempre daría similitud=1.00 ya que usa un único vector)')
    print()

## 12. Resumen

In [ ]:
print('=' * 65)
print('  RESUMEN — BERT (bert-base-uncased)')
print('=' * 65)
print(f'  Corpus completo: {len(df_sample):,} canciones | {len(generos_validos)} géneros')
print(f'  Sin muestreo   : se procesaron TODAS las canciones')
print(f'  Embedding dim  : 768 (vector [CLS])')
print(f'  Modelo         : bert-base-uncased (inglés)')
print()
print('  Análisis realizados:')
print('  • Embeddings guardados en MongoDB (campo beto_cls)')
print(f'  • Polisemia contextual: {len(polisemia_embs)} palabras × múltiples géneros')
print('  • Búsqueda semántica con Precision@5 por género')
print('  • Masked LM: predicciones de vocabulario por género')
print('  • Similitud entre géneros con centroides BERT')
print('  • Comparación cualitativa Word2Vec vs BERT')
print()
print('  CONCLUSIONES:')
print('  • BERT asigna vectores distintos según contexto (polisemia real)')
print('  • Word2Vec asigna un vector único por palabra (estático)')
print('  • Las predicciones MLM reflejan el dominio musical en inglés')
print('  • La búsqueda semántica funciona mejor con BERT que con BoW')
print('=' * 65)